# Doctor-Hospital Assignment — Interactive Demo

Compare the optimal Hungarian assignment with a randomized greedy baseline using your own input.

**Run the cells from top to bottom in the same kernel.** Section 2 collects input and immediately displays its summary; the last section runs the comparison. After changing input, rerun the input cell and the final results cell. After restarting the kernel, rerun all cells.

## 1. Prepare preference input helpers

The functions below parse and check manual rankings or generate reproducible random rankings. For example, `2 1 3` means `H2 > H1 > H3`. Every hospital must appear exactly once.

Random mode samples a complete ranking independently for each doctor. Reusing the same seed and input sizes reproduces the same rankings.


In [23]:
from __future__ import annotations
import random
def parse_preference_entry(entry: str, hospitals: list[str]) -> list[str]:
    tokens = entry.replace(",", " ").split()
    converted = []
    for token in tokens:
        if token.isdigit():
            index = int(token) - 1
            if index < 0 or index >= len(hospitals):
                raise ValueError(f"Hospital number {token} is outside the valid range.")
            converted.append(hospitals[index])
        else:
            converted.append(token)

    if len(converted) != len(hospitals):
        raise ValueError(f"Enter exactly {len(hospitals)} hospitals.")
    if len(set(converted)) != len(converted):
        raise ValueError("Each hospital must appear exactly once.")
    if set(converted) != set(hospitals):
        raise ValueError("Preferences must contain every hospital exactly once.")
    return converted

def generate_random_preferences(
    doctors: list[str],
    hospitals: list[str],
    seed: int | None = None,
) -> dict[str, list[str]]:
    generator = random.Random(seed)
    return {
        doctor: generator.sample(hospitals, len(hospitals))
        for doctor in doctors
    }
def collect_manual_preferences(
    doctors: list[str],
    hospitals: list[str],
    input_function=input,
    output_function=print,
) -> dict[str, list[str]]:
    output_function("Hospitals: " + ", ".join(
        f"{index}={hospital}" for index, hospital in enumerate(hospitals, start=1)
    ))
    preferences = {}
    for doctor in doctors:
        while True:
            entry = input_function(
                f"Preferences for {doctor} (best to worst, separated by spaces): "
            )
            try:
                preferences[doctor] = parse_preference_entry(entry, hospitals)
                break
            except ValueError as error:
                output_function(f"Invalid preference list: {error}")
    return preferences

## 2. Collect and review the input

Enter the number of doctors and hospitals, each hospital's capacity, and the preference mode (`m` for manual or `r` for random). Random mode accepts an integer seed and defaults to 0.

The cell immediately displays the doctor/hospital counts, total capacity, spare slots, individual hospital capacities, and complete preference rankings (best to worst).

`collect_inputs()` returns `doctors`, `hospitals`, `capacities`, and `preferences`. The final line stores them in the notebook so later cells can use them directly; no additional `global` declaration is needed.


In [24]:
def print_table(title, headers, rows):
    """Print an aligned table using only the Python standard library."""
    rows = [[str(value) for value in row] for row in rows]
    widths = [
        max(len(header), max((len(row[i]) for row in rows), default=0))
        for i, header in enumerate(headers)
    ]
    separator = "-+-".join("-" * width for width in widths)
    print(f"\n{title}")
    print(" | ".join(header.ljust(width) for header, width in zip(headers, widths)))
    print(separator)
    for row in rows:
        print(" | ".join(value.ljust(width) for value, width in zip(row, widths)))


def read_positive_integer(prompt: str) -> int:
    """Read a positive integer, repeating until the input is valid."""
    while True:
        try:
            value = int(input(prompt))
            if value <= 0:
                raise ValueError
            return value
        except ValueError:
            print("Please enter a positive integer.")

def read_capacities(hospitals: list[str], doctor_count: int) -> dict[str, int]:
    """Read capacities and repeat the complete entry if total capacity is low."""
    while True:
        capacities = {}
        print("\nEnter a non-negative integer capacity for each hospital.")
        for hospital in hospitals:
            while True:
                try:
                    capacity = int(input(f"Capacity for {hospital}: "))
                    if capacity < 0:
                        raise ValueError
                    capacities[hospital] = capacity
                    break
                except ValueError:
                    print("Capacity must be a non-negative integer.")

        total_capacity = sum(capacities.values())
        if total_capacity >= doctor_count:
            return capacities
        print(
            f"Total capacity is {total_capacity}, but {doctor_count} doctors "
            "need assignments. Please enter all capacities again."
        )
def collect_preferences(
    doctors: list[str], hospitals: list[str]
) -> dict[str, list[str]]:
    """Let the user select manual or reproducible random preferences."""
    while True:
        mode = input("\nPreference mode: manual (m) or random demo (r)? ").strip().lower()
        if mode in {"m", "manual"}:
            return collect_manual_preferences(doctors, hospitals)
        if mode in {"r", "random"}:
            seed_entry = input("Random seed [0]: ").strip()
            try:
                seed = int(seed_entry) if seed_entry else 0
            except ValueError:
                print("Random seed must be an integer.")
                continue
            return generate_random_preferences(doctors, hospitals, seed=seed)
        print("Enter m for manual input or r for a random demo.")
def print_preferences(preferences: dict[str, list[str]]) -> None:
    """Display complete preference rankings."""
    print_table("Doctor preferences (best to worst)", ["Doctor", "Preference order"], [
        [doctor, " > ".join(hospitals)]
        for doctor, hospitals in preferences.items()
    ])
def print_assignment(title: str, assignment: dict[str, dict[str, str]]) -> None:
    """Display one assignment in doctor order."""
    print(f"\n{title}")
    print("-" * 50)
    for doctor, result in assignment.items():
        print(f"{doctor} -> {result['hospital']}")
def collect_inputs():
    doctor_count = read_positive_integer("Number of doctors: ")
    hospital_count = read_positive_integer("Number of hospitals: ")

    doctors = [f"D{index}" for index in range(1, doctor_count + 1)]
    hospitals = [f"H{index}" for index in range(1, hospital_count + 1)]

    capacities = read_capacities(hospitals, doctor_count)
    preferences = collect_preferences(doctors, hospitals)

    print_table("Input summary", ["Input", "Value"], [
        ["Doctors", doctor_count],
        ["Hospitals", hospital_count],
        ["Total capacity", sum(capacities.values())],
        ["Spare slots", sum(capacities.values()) - doctor_count],
    ])
    print_table("Hospital capacities", ["Hospital", "Capacity"], list(capacities.items()))
    print_preferences(preferences)

    return doctors, hospitals, capacities, preferences


doctors, hospitals, capacities, preferences = collect_inputs()



Enter a non-negative integer capacity for each hospital.

Input summary
Input          | Value
---------------+------
Doctors        | 10   
Hospitals      | 3    
Total capacity | 10   
Spare slots    | 0    

Hospital capacities
Hospital | Capacity
---------+---------
H1       | 3       
H2       | 3       
H3       | 4       

Doctor preferences (best to worst)
Doctor | Preference order
-------+-----------------
D1     | H1 > H3 > H2    
D2     | H1 > H2 > H3    
D3     | H2 > H3 > H1    
D4     | H1 > H2 > H3    
D5     | H2 > H3 > H1    
D6     | H3 > H2 > H1    
D7     | H3 > H1 > H2    
D8     | H2 > H1 > H3    
D9     | H1 > H3 > H2    
D10    | H3 > H1 > H2    


## 3. Validate the input

Every doctor must rank every hospital exactly once. Capacities must be non-negative integers, and total capacity must be at least the number of doctors.

This cell defines `validate_inputs()`, which is called by the solvers before computing an assignment.


In [25]:
def validate_inputs(
    preferences: dict[str, list[str]],
    capacities: dict[str, int],
) -> None:
    if not isinstance(preferences, dict):
        raise TypeError("Preferences must be a dictionary.")
    if not isinstance(capacities, dict):
        raise TypeError("Capacities must be a dictionary.")
    if not preferences:
        raise ValueError("At least one doctor must be provided.")
    if not capacities:
        raise ValueError("At least one hospital must be provided.")

    hospital_names = set(capacities)
    for hospital, capacity in capacities.items():
        if not isinstance(hospital, str) or not hospital.strip():
            raise ValueError("Every hospital name must be a non-empty string.")
        if isinstance(capacity, bool) or not isinstance(capacity, int):
            raise TypeError(f"Capacity for {hospital!r} must be an integer.")
        if capacity < 0:
            raise ValueError(f"Capacity for {hospital!r} cannot be negative.")

    for doctor, ranked_hospitals in preferences.items():
        if not isinstance(doctor, str) or not doctor.strip():
            raise ValueError("Every doctor name must be a non-empty string.")
        if not isinstance(ranked_hospitals, list):
            raise TypeError(f"Preferences for {doctor!r} must be a list.")
        if len(ranked_hospitals) != len(capacities):
            raise ValueError(f"{doctor!r} must rank every hospital exactly once.")
        if any(not isinstance(name, str) or not name.strip() for name in ranked_hospitals):
            raise ValueError(f"All hospitals ranked by {doctor!r} must have valid names.")
        if len(set(ranked_hospitals)) != len(ranked_hospitals):
            raise ValueError(f"Preferences for {doctor!r} contain duplicate hospitals.")
        if set(ranked_hospitals) != hospital_names:
            missing = sorted(hospital_names - set(ranked_hospitals))
            unknown = sorted(set(ranked_hospitals) - hospital_names)
            raise ValueError(
                f"Preferences for {doctor!r} do not match the hospital list. "
                f"Missing: {missing}; unknown: {unknown}."
            )

    total_capacity = sum(capacities.values())
    if total_capacity < len(preferences):
        raise ValueError(
            "Total hospital capacity must be at least the number of doctors. "
            f"Doctors: {len(preferences)}; total capacity: {total_capacity}."
        )

## 4. Find the optimal assignment: Hungarian algorithm

The objective is to **minimize total rank cost**, where rank 1 is a doctor's first choice. Each doctor receives one hospital, and no hospital exceeds its capacity.

The functions below implement this pipeline:

1. `create_slots()` expands each hospital into individual capacity slots (capacity 3 becomes 3 slots).
2. `create_rank_lookup()` and `build_cost_matrix()` convert preferences into a doctor-by-slot cost matrix.
3. `hungarian_algorithm()` finds a minimum-cost assignment; `solve_hungarian()` maps slots back to hospitals.

Extra slots may remain unused. Multiple equally optimal assignments can exist.


In [26]:
def create_slots(capacities: dict[str, int]) -> list[dict[str, str]]:
    """Expand each hospital capacity into one-to-one assignment slots."""
    return [
        {"hospital": hospital, "slot": f"{hospital}_{slot_number}"}
        for hospital, capacity in capacities.items()
        for slot_number in range(1, capacity + 1)
    ]


def create_rank_lookup(
    preferences: dict[str, list[str]],
) -> dict[str, dict[str, int]]:
    """Convert ordered preference lists to one-based rank costs."""
    return {
        doctor: {
            hospital: rank
            for rank, hospital in enumerate(preference_list, start=1)
        }
        for doctor, preference_list in preferences.items()
    }


def build_cost_matrix(
    doctors: list[str],
    slots: list[dict[str, str]],
    rank_lookup: dict[str, dict[str, int]],
) -> list[list[int]]:
    """Build a doctor-by-slot rank cost matrix."""
    return [
        [rank_lookup[doctor][slot["hospital"]] for slot in slots]
        for doctor in doctors
    ]


def hungarian_algorithm(cost_matrix: list[list[int]]) -> list[int]:
    """Return the minimum-cost slot index for every matrix row."""
    if not cost_matrix or not cost_matrix[0]:
        raise ValueError("Cost matrix cannot be empty.")
    if any(len(row) != len(cost_matrix[0]) for row in cost_matrix):
        raise ValueError("Cost matrix must be rectangular.")

    row_count = len(cost_matrix)
    column_count = len(cost_matrix[0])
    if row_count > column_count:
        raise ValueError(
            "Number of hospital slots must be at least the number of doctors."
        )

    row_potential = [0] * (row_count + 1)
    column_potential = [0] * (column_count + 1)
    matched_row = [0] * (column_count + 1)
    previous_column = [0] * (column_count + 1)

    for row in range(1, row_count + 1):
        matched_row[0] = row
        current_column = 0
        minimum_reduced_cost = [float("inf")] * (column_count + 1)
        used = [False] * (column_count + 1)

        while True:
            used[current_column] = True
            current_row = matched_row[current_column]
            delta = float("inf")
            next_column = 0

            for column in range(1, column_count + 1):
                if not used[column]:
                    reduced_cost = (
                        cost_matrix[current_row - 1][column - 1]
                        - row_potential[current_row]
                        - column_potential[column]
                    )
                    if reduced_cost < minimum_reduced_cost[column]:
                        minimum_reduced_cost[column] = reduced_cost
                        previous_column[column] = current_column
                    if minimum_reduced_cost[column] < delta:
                        delta = minimum_reduced_cost[column]
                        next_column = column

            for column in range(column_count + 1):
                if used[column]:
                    row_potential[matched_row[column]] += delta
                    column_potential[column] -= delta
                else:
                    minimum_reduced_cost[column] -= delta

            current_column = next_column
            if matched_row[current_column] == 0:
                break

        while True:
            next_column = previous_column[current_column]
            matched_row[current_column] = matched_row[next_column]
            current_column = next_column
            if current_column == 0:
                break

    assignment = [-1] * row_count
    for column in range(1, column_count + 1):
        if matched_row[column] != 0:
            assignment[matched_row[column] - 1] = column - 1
    return assignment


def solve_hungarian(
    preferences: dict[str, list[str]],
    capacities: dict[str, int],
) -> dict[str, dict[str, str]]:
    """Validate and solve a complete doctor-hospital assignment problem."""
    validate_inputs(preferences, capacities)
    doctors = list(preferences)
    slots = create_slots(capacities)
    rank_lookup = create_rank_lookup(preferences)
    cost_matrix = build_cost_matrix(doctors, slots, rank_lookup)
    slot_indices = hungarian_algorithm(cost_matrix)

    return {
        doctor: {
            "hospital": slots[slot_index]["hospital"],
            "slot": slots[slot_index]["slot"],
        }
        for doctor, slot_index in zip(doctors, slot_indices)
    }

## 5. Build the randomized greedy baseline

Greedy processes doctors one at a time and gives each doctor their highest-ranked hospital with a remaining slot. It does not reconsider earlier assignments, so the result depends on doctor order.

`run_randomized_greedy()` shuffles that order for each trial while keeping preferences and capacities fixed. The final experiment uses **100 trials with seed 0**, so the comparison is reproducible.


In [27]:
def _greedy_assignment(
    preferences: dict[str, list[str]],
    capacities: dict[str, int],
    doctor_order: list[str],
) -> dict[str, dict[str, str]]:
    remaining_capacity = capacities.copy()
    assignment = {}
    for doctor in doctor_order:
        for hospital in preferences[doctor]:
            if remaining_capacity[hospital] > 0:
                assignment[doctor] = {"hospital": hospital}
                remaining_capacity[hospital] -= 1
                break
    return assignment


def greedy_assignment(
    preferences: dict[str, list[str]],
    capacities: dict[str, int],
    doctor_order: list[str] | None = None,
) -> dict[str, dict[str, str]]:
    validate_inputs(preferences, capacities)
    order = list(preferences) if doctor_order is None else list(doctor_order)
    if set(order) != set(preferences) or len(order) != len(preferences):
        raise ValueError("Doctor order must contain every doctor exactly once.")
    return _greedy_assignment(preferences, capacities, order)


def run_randomized_greedy(
    preferences: dict[str, list[str]],
    capacities: dict[str, int],
    trials: int = 100,
    seed: int | None = 0,
) -> list[dict[str, dict[str, str]]]:
    if isinstance(trials, bool) or not isinstance(trials, int) or trials <= 0:
        raise ValueError("Trials must be a positive integer.")
    validate_inputs(preferences, capacities)
    generator = random.Random(seed)
    doctors = list(preferences)
    assignments = []
    for _ in range(trials):
        order = doctors.copy()
        generator.shuffle(order)
        assignments.append(_greedy_assignment(preferences, capacities, order))
    return assignments

## 6. Define the evaluation metrics

| Metric | Meaning | Preferred direction |
| :--- | :--- | :--- |
| Total rank cost | Sum of all assigned ranks | Lower |
| Average rank | Total rank cost divided by the number of doctors | Lower |
| First-choice rate | Fraction assigned to rank 1 | Higher |
| Top-three rate | Fraction assigned to ranks 1–3 | Higher |
| Worst assigned rank | Largest assigned rank | Lower |
| Rank distribution | Number of doctors receiving each rank | Inspect the distribution |

`evaluate_assignment()` scores one assignment. `summarize_trials()` reports the greedy mean metrics and the best/worst **total rank costs** across trials.

The Hungarian algorithm optimizes total rank cost, not each individual metric. Its first-choice rate or worst assigned rank is not guaranteed to be better than greedy's. With fewer than three hospitals, every valid assignment is in the top three.


In [28]:
from collections import Counter
from statistics import mean
def evaluate_assignment(
    doctors: list[str],
    assignment: dict[str, dict[str, str]],
    preferences: dict[str, list[str]],
) -> dict[str, object]:
    """Evaluate a complete assignment using rank-based metrics."""
    if len(doctors) != len(set(doctors)):
        raise ValueError("Doctor list cannot contain duplicate names.")
    if set(doctors) != set(preferences):
        raise ValueError("Doctor list must match the preference dictionary.")
    if set(assignment) != set(doctors) or len(assignment) != len(doctors):
        raise ValueError("Every doctor must have exactly one assignment.")

    rank_lookup = create_rank_lookup(preferences)
    assigned_ranks = {}
    for doctor in doctors:
        hospital = assignment[doctor].get("hospital")
        if hospital not in rank_lookup[doctor]:
            raise ValueError(
                f"Assignment for {doctor!r} contains an unknown hospital."
            )
        assigned_ranks[doctor] = rank_lookup[doctor][hospital]
    ranks = list(assigned_ranks.values())
    total_cost = sum(ranks)
    first_choice_rate = sum(rank == 1 for rank in ranks)/len(doctors)
    top_three_count = sum(rank <= 3 for rank in ranks)

    return {
        "total_cost": total_cost,
        "average_rank": total_cost / len(doctors),
        "first_choice_rate": first_choice_rate,
        "top_three_rate": top_three_count / len(doctors),
        "worst_assigned_rank": max(ranks),
        "rank_distribution": dict(sorted(Counter(ranks).items())),
        "assigned_ranks": assigned_ranks,
    }


def summarize_trials(trial_metrics: list[dict[str, object]]) -> dict[str, float]:
    """Summarize repeated randomized greedy trials."""
    if not trial_metrics:
        raise ValueError("At least one trial is required.")
    costs = [float(metrics["total_cost"]) for metrics in trial_metrics]
    return {
        "mean_total_cost": mean(costs),
        "best_total_cost": min(costs),
        "worst_total_cost": max(costs),
        "mean_average_rank": mean(
            float(metrics["average_rank"]) for metrics in trial_metrics
        ),
        "mean_first_choice_rate": mean(
            float(metrics["first_choice_rate"]) for metrics in trial_metrics
        ),
        "mean_top_three_rate": mean(
            float(metrics["top_three_rate"]) for metrics in trial_metrics
        ),
        "mean_worst_assigned_rank": mean(
            float(metrics["worst_assigned_rank"]) for metrics in trial_metrics
        ),
    }


## 7. Run the experiment and review the results

The cell below runs Hungarian once and randomized greedy 100 times (seed 0) on the same input. The display helpers only format the results.

Read the **performance comparison** first: greedy values in that table are means across trials, including the mean of each trial's worst rank. The best and worst greedy total costs are shown separately.

The reported cost reduction is `(greedy mean cost − Hungarian cost) / greedy mean cost`. Assignment details, capacity usage, and rank distribution below the comparison refer to the Hungarian solution. Rates are displayed as percentages; mean values are rounded to three decimal places for readability.

If `preferences` is undefined, rerun Section 2 in the current kernel before running this cell.


In [29]:
def print_results(doctors, capacities, assignment, metrics, summary, trials, seed):
    """Format existing results without changing the assignment or metrics."""
    print("\n" + "=" * 72)
    print("DOCTOR-HOSPITAL ASSIGNMENT RESULTS")
    print(f"Randomized greedy: {trials:,} trials | Seed: {seed}")
    print("=" * 72)
    print_table("Performance comparison", ["Metric", "Hungarian", "Greedy mean"], [
        ["Total rank cost (lower is better)", metrics["total_cost"],
         f"{summary['mean_total_cost']:.3f}"],
        ["Average rank (lower is better)", f"{metrics['average_rank']:.3f}",
         f"{summary['mean_average_rank']:.3f}"],
        ["First-choice rate (higher is better)", f"{metrics['first_choice_rate']:.1%}",
         f"{summary['mean_first_choice_rate']:.1%}"],
        ["Top-three rate (higher is better)", f"{metrics['top_three_rate']:.1%}",
         f"{summary['mean_top_three_rate']:.1%}"],
        ["Worst assigned rank (lower is better)", metrics["worst_assigned_rank"],
         f"{summary['mean_worst_assigned_rank']:.3f}"],
    ])
    print(f"\nGreedy total rank cost: best = {summary['best_total_cost']:.0f}"
          f" | mean = {summary['mean_total_cost']:.3f}"
          f" | worst = {summary['worst_total_cost']:.0f}")
    reduction = (summary["mean_total_cost"] - metrics["total_cost"]) / summary["mean_total_cost"]
    print(f"Hungarian cost reduction vs. greedy mean: {reduction:.1%}")

    print_table("Hungarian assignment", ["Doctor", "Hospital", "Assigned rank"], [
        [doctor, assignment[doctor]["hospital"], metrics["assigned_ranks"][doctor]]
        for doctor in doctors
    ])
    occupancy = Counter(result["hospital"] for result in assignment.values())
    print_table("Hospital capacity usage (Hungarian)",
                ["Hospital", "Capacity", "Assigned", "Remaining"], [
        [hospital, capacity, occupancy[hospital], capacity - occupancy[hospital]]
        for hospital, capacity in capacities.items()
    ])
    print_table("Assigned rank distribution (Hungarian)", ["Rank", "Doctors", "Share"], [
        [rank, count, f"{count / len(doctors):.1%}"]
        for rank, count in metrics["rank_distribution"].items()
    ])


def main() -> None:
    hungarian_assignment = solve_hungarian(preferences, capacities)

    hungarian_metrics = evaluate_assignment(
        doctors, hungarian_assignment, preferences
    )

    greedy_assignments = run_randomized_greedy(
        preferences, capacities, trials=100, seed=0
    )

    greedy_metrics = [
        evaluate_assignment(doctors, assignment, preferences)
        for assignment in greedy_assignments
    ]

    print_results(
        doctors, capacities, hungarian_assignment, hungarian_metrics,
        summarize_trials(greedy_metrics), trials=100, seed=0,
    )


if __name__ == "__main__":
    main()



DOCTOR-HOSPITAL ASSIGNMENT RESULTS
Randomized greedy: 100 trials | Seed: 0

Performance comparison
Metric                                | Hungarian | Greedy mean
--------------------------------------+-----------+------------
Total rank cost (lower is better)     | 11        | 11.670     
Average rank (lower is better)        | 1.100     | 1.167      
First-choice rate (higher is better)  | 90.0%     | 86.9%      
Top-three rate (higher is better)     | 100.0%    | 100.0%     
Worst assigned rank (lower is better) | 2         | 2.360      

Greedy total rank cost: best = 11 | mean = 11.670 | worst = 13
Hungarian cost reduction vs. greedy mean: 5.7%

Hungarian assignment
Doctor | Hospital | Assigned rank
-------+----------+--------------
D1     | H1       | 1            
D2     | H1       | 1            
D3     | H2       | 1            
D4     | H1       | 1            
D5     | H2       | 1            
D6     | H3       | 1            
D7     | H3       | 1            
D8     | H2  